Connected to Python 3.14.7

In [ ]:
import polars as pl
import requests #new for reading in JSON
import io       #new for reading in JSON
import numpy as np
from lets_plot import *

LetsPlot.setup_html(isolated_frame=True)

In [ ]:
# Learn morea about Code Cells: https://quarto.org/docs/reference/cells/cells-jupyter.html

# Include and execute your code here
url = "https://raw.githubusercontent.com/byuidatascience/data4missing/master/data-raw/flights_missing/flights_missing.json"
response = requests.get(url)
df = pl.read_json(io.BytesIO(response.content))
df.head(5)

airport_code,airport_name,month,year,num_of_flights_total,num_of_delays_carrier,num_of_delays_late_aircraft,num_of_delays_nas,num_of_delays_security,num_of_delays_weather,num_of_delays_total,minutes_delayed_carrier,minutes_delayed_late_aircraft,minutes_delayed_nas,minutes_delayed_security,minutes_delayed_weather,minutes_delayed_total
str,str,str,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""ATL""","""Atlanta, GA: Hartsfield-Jackso…","""January""",2005,35048,"""1500+""",-999,4598,10,448,8355,116423,104415,207467,297,36931,465533
"""DEN""","""Denver, CO: Denver Internation…","""January""",2005,12687,"""1041""",928,935,11,233,3153,53537,70301,36817,363,21779,182797
"""IAD""","""""","""January""",2005,12381,"""414""",1058,895,4,61,2430,null,70919,35660,208,4497,134881
"""ORD""","""Chicago, IL: Chicago O'Hare In…","""January""",2005,28194,"""1197""",2255,5415,5,306,9178,88691,160811,364382,151,24859,638894
"""SAN""","""San Diego, CA: San Diego Inter…","""January""",2005,7283,"""572""",680,638,7,56,1952,27436,38445,21127,218,4326,91552


In [ ]:
# Include and execute your code here
df = df.with_columns(
pl.when(pl.col("month").is_in(["January", "Febuary", "March"])).then(pl.lit("Winter"))
.when(pl.col("month").is_in(["April", "May", "June"])).then(pl.lit("Spring"))
.when(pl.col("month").is_in(["July", "August", "September"])).then(pl.lit("Summer"))
.when(pl.col("month").is_in(["October", "November", "December"])).then(pl.lit("Fall"))
.otherwise(pl.lit("Unknown"))
.alias("season")
)

In [ ]:
# Include and execute your code here
df.filter(
    (pl.col("num_of_delays_weather") != "-999") &
    (pl.col("num_of_delays_weather") != "1500") &
    (pl.col("num_of_delays_nas") != "-999") &
    (pl.col("num_of_delays_nas") != "1500") &
    (pl.col("num_of_delays_late_aircraft") != "-999") &
    (pl.col("num_of_delays_late_aircraft") != "1500")
)

.(pl.col("^num_of_.*$").cast(pl.Float64) / 1000)

SyntaxError: invalid syntax (<ipython-input-4-79c0ecd02b33>, line 11)

In [ ]:
# Include and execute your code here
# Questão 3
df = df.with_columns(
    (
        pl.col("num_of_delays_weather") + 
(pl.col("num_of_delays_late_aircraft") * 0.30) + 
 pl.when(pl.col("month").is_in(["April", "May", "June", "July", "August"]))
  .then(pl.col("num_of_delays_nas") * 0.40)
        .otherwise(pl.col("num_of_delays_nas") * 0.65)
    ).alias("total_weather_delays")
)


df.head(5)

airport_code,airport_name,month,year,num_of_flights_total,num_of_delays_carrier,num_of_delays_late_aircraft,num_of_delays_nas,num_of_delays_security,num_of_delays_weather,num_of_delays_total,minutes_delayed_carrier,minutes_delayed_late_aircraft,minutes_delayed_nas,minutes_delayed_security,minutes_delayed_weather,minutes_delayed_total,season,total_weather_delays
str,str,str,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,f64
"""ATL""","""Atlanta, GA: Hartsfield-Jackso…","""January""",2005,35048,"""1500+""",-999,4598,10,448,8355,116423,104415,207467,297,36931,465533,"""Winter""",3137.0
"""DEN""","""Denver, CO: Denver Internation…","""January""",2005,12687,"""1041""",928,935,11,233,3153,53537,70301,36817,363,21779,182797,"""Winter""",1119.15
"""IAD""","""""","""January""",2005,12381,"""414""",1058,895,4,61,2430,null,70919,35660,208,4497,134881,"""Winter""",960.15
"""ORD""","""Chicago, IL: Chicago O'Hare In…","""January""",2005,28194,"""1197""",2255,5415,5,306,9178,88691,160811,364382,151,24859,638894,"""Winter""",4502.25
"""SAN""","""San Diego, CA: San Diego Inter…","""January""",2005,7283,"""572""",680,638,7,56,1952,27436,38445,21127,218,4326,91552,"""Winter""",674.7


In [ ]:
# Include and execute your code here

# Questão 4
grafic = (
ggplot(df, aes(x='season', y='total_weather_delays')) + 
geom_boxplot() + labs(title="Distribution of Weather Delays by Season", x="Season of the Year", y="Total Delays in thousands"))

grafic.show()

In [ ]:
# Include and execute your code here

# Questão 4
grafic = (
ggplot(df, aes(x='season', y='total_weather_delays')) + 
geom_boxplot() + labs(title="Distribution of Weather Delays by Season", x="Season of the Year", y="Total Delays in thousands"))

grafic.show()